# Data Cleaning: Keep rows with real poster images

This notebook filters `MovieGenre.csv` down to rows that can be matched to a real image file from the Kaggle dataset.

Goal:
- keep rows with non-null `Genre`
- resolve each row to an existing image file
- export a cleaned CSV for downstream experiments
- estimate whether cleaned data size supports CNN fine-tune, CNN scratch, and ViT fine-tune

In [ ]:
# 1) Imports and project paths
from __future__ import annotations

import os
import re
import subprocess
import sys
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

import pandas as pd


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "MovieGenre.csv").is_file() and (base / "README.md").is_file():
            return base
    raise RuntimeError("Could not find ieor142b project root.")


ROOT = find_project_root()
CSV_PATH = ROOT / "MovieGenre.csv"
OUT_DIR = ROOT / "cleaned"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("CSV path:", CSV_PATH)

# Optional override if you already downloaded the Kaggle dataset elsewhere.
# Example:
# os.environ["KAGGLE_DATASET_PATH"] = "/absolute/path/to/movie-genre-from-its-poster"
KAGGLE_DATASET_PATH = os.environ.get("KAGGLE_DATASET_PATH")


def ensure_kagglehub():
    try:
        import kagglehub  # type: ignore
        return kagglehub
    except ImportError:
        print("kagglehub not found. Installing into current kernel...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub", "-q"])
        import kagglehub  # type: ignore
        return kagglehub

In [ ]:
# 2) Build image pool from (A) Poster URL downloads and/or (B) Kaggle fallback
# Recommended: keep both enabled so dead links can still match local Kaggle images.

DOWNLOAD_FROM_POSTER_URLS = True
USE_KAGGLE_FALLBACK = True
MAX_DOWNLOAD_ROWS = None  # full sweep
DOWNLOAD_TIMEOUT_SEC = 6
DOWNLOAD_RETRIES = 0
DOWNLOAD_SLEEP_SEC = 0.0
PROGRESS_EVERY = 1000

# Early stop once we have enough strict imdbId-based images.
ENABLE_EARLY_STOP_STRICT = True
TARGET_STRICT_MATCHES = 10000  # set to 10000 if you want a larger target

DOWNLOAD_DIR = OUT_DIR / "downloaded_posters"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)


def download_poster(url: str, out_path: Path) -> tuple[bool, str | None]:
    if out_path.exists() and out_path.stat().st_size > 0:
        return True, "cached"
    if not isinstance(url, str) or not url.strip():
        return False, "empty_url"

    # Some hosts reject default Python UA; use browser-like UA.
    req = Request(url.strip(), headers={"User-Agent": "Mozilla/5.0"})

    for attempt in range(DOWNLOAD_RETRIES + 1):
        try:
            with urlopen(req, timeout=DOWNLOAD_TIMEOUT_SEC) as r:
                content = r.read()
            if not content:
                return False, "empty_body"
            out_path.write_bytes(content)
            return True, None
        except HTTPError as e:
            if e.code in {403, 404, 410}:
                return False, f"http_{e.code}"
            last_err = f"http_{e.code}"
        except URLError as e:
            last_err = f"urlerror_{str(e.reason)}"
        except Exception as e:
            last_err = f"other_{type(e).__name__}"

        if attempt < DOWNLOAD_RETRIES:
            time.sleep(DOWNLOAD_SLEEP_SEC)

    return False, last_err


image_files = []

if DOWNLOAD_FROM_POSTER_URLS:
    src_df = pd.read_csv(CSV_PATH, encoding="latin-1").dropna(subset=["Genre"]).reset_index(drop=True)
    if MAX_DOWNLOAD_ROWS is not None:
        src_df = src_df.head(MAX_DOWNLOAD_ROWS).copy()

    ok_count = 0
    fail_count = 0
    fail_reasons: dict[str, int] = {}
    strict_match_count = 0
    processed_rows = 0
    stop_reason = "exhausted_rows"

    for i, (_, row) in enumerate(src_df.iterrows(), start=1):
        processed_rows = i
        try:
            imdb_key = int(row["imdbId"])
        except Exception:
            fail_count += 1
            fail_reasons["bad_imdbId"] = fail_reasons.get("bad_imdbId", 0) + 1
            continue

        out_path = DOWNLOAD_DIR / f"{imdb_key}.jpg"
        success, err = download_poster(row.get("Poster", ""), out_path)
        if success:
            ok_count += 1
            strict_match_count += 1
        else:
            fail_count += 1
            err_key = err or "unknown"
            fail_reasons[err_key] = fail_reasons.get(err_key, 0) + 1

        if PROGRESS_EVERY and (i % PROGRESS_EVERY == 0):
            print(
                f"Processed {i}/{len(src_df)} | ok={ok_count} fail={fail_count} "
                f"strict_matches={strict_match_count}"
            )

        if ENABLE_EARLY_STOP_STRICT and strict_match_count >= TARGET_STRICT_MATCHES:
            stop_reason = "target_strict_matches_reached"
            print(
                f"Early stop triggered at row {i}: "
                f"strict_match_count={strict_match_count} >= {TARGET_STRICT_MATCHES}"
            )
            break

    downloaded_images = [p for p in DOWNLOAD_DIR.glob("*.jpg") if p.stat().st_size > 0]
    image_files.extend(downloaded_images)

    print(f"Poster URL download mode: processed={processed_rows} of {len(src_df)}")
    print(f"Stop reason: {stop_reason}")
    print(f"Downloaded/cached OK: {ok_count} | failed: {fail_count}")
    print(f"Strict imdbId match count: {strict_match_count}")
    if fail_reasons:
        top_reasons = sorted(fail_reasons.items(), key=lambda x: x[1], reverse=True)[:10]
        print("Top failure reasons:", top_reasons)

if USE_KAGGLE_FALLBACK:
    if KAGGLE_DATASET_PATH:
        kaggle_path = Path(KAGGLE_DATASET_PATH).expanduser().resolve()
        if not kaggle_path.exists():
            raise FileNotFoundError(f"KAGGLE_DATASET_PATH does not exist: {kaggle_path}")
        print("Using local dataset path from KAGGLE_DATASET_PATH:", kaggle_path)
    else:
        kagglehub = ensure_kagglehub()
        kaggle_path = Path(kagglehub.dataset_download("neha1703/movie-genre-from-its-poster"))
        print("Kaggle dataset root:", kaggle_path)

    image_exts = {".jpg", ".jpeg", ".png", ".webp"}
    kaggle_images = [p for p in kaggle_path.rglob("*") if p.suffix.lower() in image_exts]
    image_files.extend(kaggle_images)
    print("Kaggle fallback images discovered:", len(kaggle_images))

# Deduplicate full paths
image_files = list({p.resolve(): p for p in image_files}.values())
print("Total pooled image files:", len(image_files))

if not image_files:
    raise RuntimeError(
        "No image files available. Check network access for poster downloads and/or Kaggle path."
    )

In [ ]:
# 3) Build fast lookup maps for image resolution
by_name: dict[str, Path] = {}
by_numeric_stem: dict[int, Path] = {}

for p in image_files:
    by_name.setdefault(p.name, p)
    stem = p.stem
    if stem.isdigit():
        by_numeric_stem.setdefault(int(stem), p)

print("Unique filenames:", len(by_name))
print("Numeric stems:", len(by_numeric_stem))
print("Example files:", [p.name for p in image_files[:8]])

In [ ]:
# 4) Load CSV and match each row to a real image file
raw_df = pd.read_csv(CSV_PATH, encoding="latin-1")
df = raw_df.dropna(subset=["Genre"]).reset_index(drop=True).copy()
df["row_id"] = df.index

print("Raw rows:", len(raw_df))
print("After dropna(Genre):", len(df))


def poster_basename(url: str) -> str | None:
    if not isinstance(url, str) or not url.strip():
        return None
    parsed = urlparse(url)
    name = Path(parsed.path).name
    if not name:
        return None
    # Normalize accidental query artifacts, e.g., foo.jpg?x=1
    name = re.sub(r"\?.*$", "", name)
    return name or None


def resolve_image(row: pd.Series) -> tuple[Path | None, str | None]:
    # Strategy 1: imdbId.jpg
    imdb_id = row.get("imdbId")
    try:
        imdb_key = int(imdb_id)
        if imdb_key in by_numeric_stem:
            return by_numeric_stem[imdb_key], "imdbId"
    except Exception:
        pass

    # Strategy 2: row_id.jpg (some datasets are indexed this way)
    row_key = int(row["row_id"])
    if row_key in by_numeric_stem:
        return by_numeric_stem[row_key], "row_id"

    # Strategy 3: exact filename from Poster URL basename
    poster_name = poster_basename(row.get("Poster", ""))
    if poster_name and poster_name in by_name:
        return by_name[poster_name], "poster_url_name"

    return None, None


resolved = df.apply(resolve_image, axis=1, result_type="expand")
df["image_path"] = resolved[0].map(lambda x: str(x) if x is not None else None)
df["image_match_source"] = resolved[1]
df["image_exists"] = df["image_path"].notna()

match_stats = df["image_match_source"].value_counts(dropna=False)
print("\nMatch source counts:")
print(match_stats)
print("\nRows with matched image:", int(df["image_exists"].sum()))

In [ ]:
# 5) Export strict cleaned table with portable relative image paths


def attach_relative_poster_paths(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy().reset_index(drop=True)
    out["image_path"] = out["imdbId"].astype(int).map(
        lambda i: f"cleaned/downloaded_posters/{i}.jpg"
    )
    out["image_exists"] = True
    return out


clean_df_all = df[df["image_exists"]].copy().reset_index(drop=True)
clean_df = clean_df_all[clean_df_all["image_match_source"] == "imdbId"].copy().reset_index(drop=True)
clean_df = attach_relative_poster_paths(clean_df)

# Optional working snapshot (ignored by git).
OUT_CSV = OUT_DIR / "MovieGenre_clean_with_images.csv"
clean_df.to_csv(OUT_CSV, index=False, encoding="latin-1")

print("Saved strict working CSV:", OUT_CSV)
print("strict rows (imdbId only):", len(clean_df))
print(
    "strict match sources:",
    clean_df["image_match_source"].value_counts(dropna=False).to_dict(),
)
clean_df[["imdbId", "Title", "Genre", "image_match_source", "image_path"]].head(10)

In [ ]:
# 6) Feasibility check for planned experiments (canonical strict subset)
# Approximate split sizes if you keep 80/10/10 (same ratios as splits_cleaned)
n = len(clean_df)
train_n = int(round(n * 0.8))
val_n = int(round(n * 0.1))
test_n = n - train_n - val_n

genre_counts = clean_df["Genre"].str.split("|").explode().value_counts()
rare_genres = genre_counts[genre_counts < 20]

print(f"Total cleaned rows: {n}")
print(f"Approx split -> train: {train_n}, val: {val_n}, test: {test_n}")
print(f"Unique genres retained: {genre_counts.index.nunique()}")
print(f"Genres with <20 samples: {len(rare_genres)}")
if len(rare_genres) > 0:
    print("\nRarest genres (<20):")
    print(rare_genres.tail(min(10, len(rare_genres))))

if n >= 20000:
    recommendation = (
        "Strong dataset size: proceed with CNN fine-tune, CNN scratch, and ViT fine-tune. "
        "Still expect scratch ViT to be expensive."
    )
elif n >= 8000:
    recommendation = (
        "Good size: prioritize CNN fine-tune + ViT fine-tune. "
        "CNN from scratch is feasible but may underperform without careful regularization."
    )
elif n >= 3000:
    recommendation = (
        "Moderate size: prioritize transfer learning only (CNN fine-tune + ViT fine-tune). "
        "Treat CNN from scratch as optional/ablation."
    )
else:
    recommendation = (
        "Small size: focus on one transfer model first (CNN fine-tune). "
        "Use ViT fine-tune only if validation trends are stable."
    )

print("\nRecommendation:")
print(recommendation)

In [ ]:
# 7) Persist canonical strict artifact + manifest
from datetime import datetime, timezone
import json


def _counts(series: pd.Series) -> dict[str, int]:
    return {str(k): int(v) for k, v in series.value_counts(dropna=False).to_dict().items()}


CANONICAL_STRICT_CSV = OUT_DIR / "MovieGenre_clean_with_images_full.csv"
CANONICAL_MANIFEST_JSON = OUT_DIR / "MovieGenre_clean_with_images_full_manifest.json"

clean_df.to_csv(CANONICAL_STRICT_CSV, index=False, encoding="latin-1")

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_csv": "MovieGenre.csv",
    "canonical_variant": "strict_imdbId_only",
    "canonical_clean_csv": str(CANONICAL_STRICT_CSV.relative_to(ROOT)),
    "rows_raw": int(len(raw_df)),
    "rows_after_dropna_genre": int(len(df)),
    "rows_strict": int(len(clean_df)),
    "unique_genres_strict": int(clean_df["Genre"].str.split("|").explode().nunique()),
    "strict_match_source_counts": _counts(clean_df["image_match_source"]),
    "splits_strict_dir": "splits_cleaned",
    "seed": 42,
    "download_config": {
        "DOWNLOAD_FROM_POSTER_URLS": bool(DOWNLOAD_FROM_POSTER_URLS),
        "USE_KAGGLE_FALLBACK": bool(USE_KAGGLE_FALLBACK),
        "MAX_DOWNLOAD_ROWS": MAX_DOWNLOAD_ROWS,
        "DOWNLOAD_TIMEOUT_SEC": int(DOWNLOAD_TIMEOUT_SEC),
        "DOWNLOAD_RETRIES": int(DOWNLOAD_RETRIES),
    },
}

with open(CANONICAL_MANIFEST_JSON, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
    f.write("\n")

print("Saved canonical strict CSV:", CANONICAL_STRICT_CSV)
print("Saved manifest JSON:", CANONICAL_MANIFEST_JSON)
manifest

In [ ]:
# 8) Generate fixed splits for canonical strict CSV
from sklearn.model_selection import train_test_split


def write_split_bundle(data: pd.DataFrame, out_dir: Path, seed: int = 42) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    train_df, temp_df = train_test_split(data, test_size=0.2, random_state=seed)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=seed)
    train_df.to_csv(out_dir / "train_rows.csv", index=False, encoding="latin-1")
    val_df.to_csv(out_dir / "val_rows.csv", index=False, encoding="latin-1")
    test_df.to_csv(out_dir / "test_rows.csv", index=False, encoding="latin-1")
    print("Wrote", out_dir, f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


SEED = 42
write_split_bundle(clean_df, ROOT / "splits_cleaned", SEED)

sample_train = pd.read_csv(ROOT / "splits_cleaned" / "train_rows.csv", encoding="latin-1", nrows=1)
print("Split columns:", list(sample_train.columns))

In [ ]:
# 9) Reuse mode: skip heavy download if canonical cleaned CSV already exists
# On future runs, execute only this cell + split generation cells if you don't need to refresh data.

REUSE_CANONICAL_IF_PRESENT = True
if REUSE_CANONICAL_IF_PRESENT and CANONICAL_STRICT_CSV.exists():
    cached_clean_df = pd.read_csv(CANONICAL_STRICT_CSV, encoding="latin-1")
    print("Loaded cached canonical strict CSV:", CANONICAL_STRICT_CSV)
    print("Rows:", len(cached_clean_df))
    print("Unique genres:", cached_clean_df["Genre"].str.split("|").explode().nunique())
else:
    print("Canonical cleaned CSV missing or reuse disabled.")
    print("Run full pipeline cells above to rebuild.")